# Global General Data Preparation

Processes every Excel file in `Files for Upload/`, flags validation issues against **Col's Checklist**, and saves working copies to `Working Files/` as `output_prefix_<original name>.csv` (utf-8-sig).

Each output CSV has one flag row at the top (before the header row) summarizing, per column, every rule that column failed and how many rows were affected. The data rows follow unchanged below the header.

Columns that reference a Porzio list of values (`PURPOSE`, `FORM`, `PRODUCT`, `COUNTRY`, `PROFILETYPE`, `RECIPIENTIDENTIFIERCOUNTRY`, `RECIPIENTIDENTIFIERTYPE`) are validated against the reference spreadsheets in this folder (`Purpose.xlsx`, `Form.xlsx`, `Product.xlsx`, `Country.xlsx`, `Profile Type.xlsx`, `Identifier Type.xlsx`), not just checked for blanks.

Two columns are also auto-filled (only when currently blank) before flagging runs:
- `COMPANYSALESREPID` — matched from `Roster.xlsx` by `TRANSACTIONINITIATORLASTNAME` + `TRANSACTIONINITIATORFIRSTNAME`.
- `COMPANYPROFILEID` — matched from `Profile Extract.xlsx` by `USNPINUMBER`.

Then these transformations are applied before flagging:
- `PRODUCT` is uppercased (so "Cabometyx" matches "CABOMETYX" in `Product.xlsx`).
- `COUNTRY` and `RECIPIENTIDENTIFIERCOUNTRY`: "USA" is converted to "US".
- `VENUECITY` / `VENUEPROVINCE` / `VENUEPOSTALCODE` / `VENUECOUNTRY` set to "Virtual" (any case) are replaced with `CITY` / `PROVINCE` / `POSTALCODE` / `COUNTRY` from the same row.
- Blank `PYMTCONTEXTINFO` is filled with `COMPANYEVENTID-TRANSACTIONDATE-ENGAGEMENTNAME` (skipped when `SECONDARYPURPOSE` has a value, since the checklist requires it blank then).
- Blank `COMMENTS` is filled with the original source file name.


In [ ]:
from pathlib import Path
import pandas as pd
import re

output_prefix = "BIOMEDCOM"

# Folder containing lookup/reference files and the script
script_folder = Path(
    r"C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\Scripts\Global General Data Preparation Files"
)

# Folder containing transparency data files
data_folder = Path(
    r"C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\US\Q3 2026\BiomedCom"
)

upload_dir = data_folder / "Files for Upload"
working_dir = data_folder / "Working Files"

lookup_paths = {
    "purpose": script_folder / "Purpose.xlsx",
    "form": script_folder / "Form.xlsx",
    "product": script_folder / "Product.xlsx",
    "country": script_folder / "Country.xlsx",
    "profile_type": script_folder / "Profile Type.xlsx",
    "identifier_type": script_folder / "Identifier Type.xlsx",
    "roster": script_folder / "Roster.xlsx",
    "profile_extract": script_folder / "Profile Extract.xlsx",
}

upload_dir.mkdir(parents=True, exist_ok=True)
working_dir.mkdir(parents=True, exist_ok=True)

print("Script folder:", repr(script_folder))
print("Data folder:", repr(data_folder))
print("Upload directory:", repr(upload_dir))
print("Working directory:", repr(working_dir))

Script folder: WindowsPath('C:/Users/cmclaren/OneDrive - Exelixis Inc/Desktop/Transparency/Scripts/Global General Data Preparation Files')
Data folder: WindowsPath('C:/Users/cmclaren/OneDrive - Exelixis Inc/Desktop/Transparency/US/Q3 2026/BiomedCom')
Upload directory: WindowsPath('C:/Users/cmclaren/OneDrive - Exelixis Inc/Desktop/Transparency/US/Q3 2026/BiomedCom/Files for Upload')
Working directory: WindowsPath('C:/Users/cmclaren/OneDrive - Exelixis Inc/Desktop/Transparency/US/Q3 2026/BiomedCom/Working Files')


In [8]:
for name, path in lookup_paths.items():
    print(name, path.exists(), path)

purpose True C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\Scripts\Global General Data Preparation Files\Purpose.xlsx
form True C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\Scripts\Global General Data Preparation Files\Form.xlsx
product True C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\Scripts\Global General Data Preparation Files\Product.xlsx
country True C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\Scripts\Global General Data Preparation Files\Country.xlsx
profile_type True C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\Scripts\Global General Data Preparation Files\Profile Type.xlsx
identifier_type True C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\Scripts\Global General Data Preparation Files\Identifier Type.xlsx
roster True C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\Scripts\Global General Data Preparation Files\Roster.xlsx
profile_extract True C:\Users\cmclare

In [9]:
def _norm_lov(value):
    """Normalize a lookup-table value for comparison: strip, drop NBSP, collapse whitespace."""
    if pd.isna(value):
        return ""
    text = str(value).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def _load_lov(path, *columns):
    df = pd.read_excel(path)
    values = set()
    for col in columns:
        if col in df.columns:
            values.update(_norm_lov(v) for v in df[col])
    values.discard("")
    return values


purpose_values = _load_lov(lookup_paths["purpose"], "PURPOSE")
form_values = _load_lov(lookup_paths["form"], "FORM")
product_values = {v.upper() for v in _load_lov(lookup_paths["product"], "PRODUCT BRAND NAME")}
profile_type_values = _load_lov(
    lookup_paths["profile_type"], "PROFILE TYPE (Individuals)", "PROFILE TYPE (Organizations)"
)
identifier_type_values = _load_lov(lookup_paths["identifier_type"], "RECIPIENT IDENTIFIER TYPE")

country_df = pd.read_excel(lookup_paths["country"])
country_codes = {_norm_lov(c).upper() for c in country_df["CODE"]}
country_codes.discard("")

lookups = {
    "purpose": purpose_values,
    "form": form_values,
    "product": product_values,
    "country": country_codes,
    "profile_type": profile_type_values,
    "identifier_type": identifier_type_values,
}

for name, values in lookups.items():
    print(f"Loaded {len(values)} valid values for {name}")


Loaded 42 valid values for purpose
Loaded 2 valid values for form
Loaded 3 valid values for product
Loaded 251 valid values for country
Loaded 127 valid values for profile_type
Loaded 48 valid values for identifier_type


In [10]:
def _header_map(columns):
    return {col.strip().upper(): col for col in columns if isinstance(col, str)}


def _clean(series):
    return series.fillna("").astype(str).str.strip()


def _norm_series(series):
    return series.fillna("").astype(str).apply(_norm_lov)


def _flag_columns_for_df(df, lookups):
    """Return a dict of column label -> summary flag message (empty string if no issues)."""
    header_map = _header_map(df.columns)
    column_message_counts = {}  # (COL_NAME, message) -> row count

    def get(col_name):
        return df[header_map[col_name]] if col_name in header_map else None

    def record(col_name, mask, message):
        if col_name not in header_map or mask is None:
            return
        mask = mask.fillna(False)
        count = int(mask.sum())
        if count:
            key = (col_name, message)
            column_message_counts[key] = column_message_counts.get(key, 0) + count

    def blank_mask(col_name):
        series = get(col_name)
        return _clean(series).eq("") if series is not None else None

    def nonblank_mask(col_name):
        series = get(col_name)
        return _clean(series).ne("") if series is not None else None

    def flag_if_blank(col_name, message="Blank value"):
        record(col_name, blank_mask(col_name), message)

    def flag_if_nonblank(col_name, message="Expected blank per checklist"):
        record(col_name, nonblank_mask(col_name), message)

    def flag_if_not_in(col_name, valid_values, message, upper=False, also_blank=True):
        series = get(col_name)
        if series is None:
            return
        cleaned = _norm_series(series)
        if upper:
            cleaned = cleaned.str.upper()
        blank = cleaned.eq("")
        not_valid = ~cleaned.isin(valid_values)
        mask = not_valid if also_blank else (not_valid & ~blank)
        record(col_name, mask, message)

    # ROWID: numerical, sequential, unique, starting at 1
    rowid = get("ROWID")
    if rowid is not None:
        n = len(df)
        cleaned = _clean(rowid)
        is_int = cleaned.str.match(r"^[0-9]+$")
        record("ROWID", ~is_int, "ROWID must be a whole number")
        dup_mask = is_int & cleaned.duplicated(keep=False)
        record("ROWID", dup_mask, "ROWID contains a duplicate value")
        expected = pd.Series([str(i) for i in range(1, n + 1)], index=df.index)
        record("ROWID", is_int & ~dup_mask & cleaned.ne(expected), "ROWID is not sequential starting at 1")

    # RECORDTYPE: always "I"
    record_type = get("RECORDTYPE")
    if record_type is not None:
        record("RECORDTYPE", _clean(record_type).str.upper().ne("I"), 'RECORDTYPE should be "I"')

    flag_if_nonblank("COMPANYTRANSACTIONID", "Should be blank per checklist")
    flag_if_blank("CUSTOMERSOURCESYSTEM")

    affiliated = get("AFFILIATEDCOMPANY")
    if affiliated is not None:
        record("AFFILIATEDCOMPANY", _clean(affiliated).ne("Exelixis Inc."), 'Expected "Exelixis Inc."')

    txn_date = get("TRANSACTIONDATE")
    if txn_date is not None:
        cleaned = _clean(txn_date)
        record("TRANSACTIONDATE", cleaned.eq(""), "Blank value")
        record("TRANSACTIONDATE", cleaned.ne("") & ~cleaned.str.match(r"^[0-9]{8}$"), "Not in YYYYMMDD format")

    flag_if_not_in("PURPOSE", lookups["purpose"], 'Not found in "Purpose" spreadsheet, or blank')

    secondary_purpose_nonblank = nonblank_mask("SECONDARYPURPOSE")
    flag_if_nonblank("SECONDARYPURPOSE", "Value present (should be blank)")

    flag_if_not_in("FORM", lookups["form"], 'Not found in "Form" spreadsheet, or blank')

    total_amount = get("TOTALAMOUNT")
    if total_amount is not None:
        cleaned = _clean(total_amount)
        record("TOTALAMOUNT", cleaned.eq(""), "Blank value")
        has_symbols = cleaned.str.contains(r"[,$€£¥]", regex=True)
        record("TOTALAMOUNT", cleaned.ne("") & has_symbols, "Contains currency symbols or commas")
        bad_numeric = cleaned.ne("") & ~has_symbols & ~cleaned.str.match(r"^-?[0-9]+(\.[0-9]+)?$")
        record("TOTALAMOUNT", bad_numeric, "Invalid numeric formatting")

    currency = get("CURRENCY")
    if currency is not None:
        record("CURRENCY", ~_clean(currency).str.upper().str.fullmatch(r"[A-Z]{3}"), "Expected a 3-letter currency code")

    tnr = get("TOTALNUMBEROFRECIPIENTS")
    if tnr is not None:
        record("TOTALNUMBEROFRECIPIENTS", _clean(tnr).ne("1"), "Expected 1 per checklist")

    for col_name in [
        "NUMBEROFCOMPANYREPRESENTATIVES", "NUMBEROFNONPROFESSIONALRECIPIENTS", "NUMBEROFNOSHOWS",
        "PRODUCT2", "CUSTOMERMASTERID", "RECIPIENTCATEGORY", "ORGANIZATIONNAME",
    ]:
        flag_if_nonblank(col_name)

    for col_name in [
        "COMPANYSALESREPID", "TRANSACTIONINITIATORFIRSTNAME", "TRANSACTIONINITIATORLASTNAME",
        "LASTNAME", "FIRSTNAME", "MIDDLENAME", "ADDRESS1", "ADDRESS2", "CITY", "PROVINCE", "POSTALCODE",
        "USNPINUMBER", "COMPANYEVENTID", "VENUECITY", "VENUEPROVINCE", "VENUECOUNTRY", "VENUEPOSTALCODE",
    ]:
        flag_if_blank(col_name)

    flag_if_not_in("PRODUCT", lookups["product"], 'Not found in "Product" spreadsheet, or blank', upper=True)

    # INDIRECTPAYMENT: blank while PAYEENAME has a value
    payeename_nonblank = nonblank_mask("PAYEENAME")
    if payeename_nonblank is not None:
        record("INDIRECTPAYMENT", blank_mask("INDIRECTPAYMENT") & payeename_nonblank,
               "Blank while PAYEENAME has a value")

    # PAYEENAME: blank while INDIRECTPAYMENT has a value
    indirect_nonblank = nonblank_mask("INDIRECTPAYMENT")
    if indirect_nonblank is not None:
        record("PAYEENAME", blank_mask("PAYEENAME") & indirect_nonblank,
               "Blank while INDIRECTPAYMENT has a value")

    # PAYEETYPE: flag if blank, or not a valid code
    flag_if_blank("PAYEETYPE")
    payee_type = get("PAYEETYPE")
    if payee_type is not None:
        cleaned = _clean(payee_type)
        valid_payee_types = {"1", "2", "3", "4", "5", "6", "7"}
        record("PAYEETYPE", cleaned.ne("") & ~cleaned.isin(valid_payee_types), "Expected a valid payee type code (1-7)")

    # MATERIALNAME / MATERIALQTY: blank if the other one has a value
    material_qty_nonblank = nonblank_mask("MATERIALQTY")
    if material_qty_nonblank is not None:
        mask = blank_mask("MATERIALNAME")
        if mask is not None:
            record("MATERIALNAME", mask & material_qty_nonblank, "Blank while MATERIALQTY has a value")

    material_name_nonblank = nonblank_mask("MATERIALNAME")
    if material_name_nonblank is not None:
        mask = blank_mask("MATERIALQTY")
        if mask is not None:
            record("MATERIALQTY", mask & material_name_nonblank, "Blank while MATERIALNAME has a value")

    flag_if_not_in("COUNTRY", lookups["country"], 'Not found in "Country" spreadsheet, or blank', upper=True)
    flag_if_not_in("PROFILETYPE", lookups["profile_type"], 'Not found in "Profile Type" spreadsheet, or blank')
    flag_if_not_in("RECIPIENTIDENTIFIERCOUNTRY", lookups["country"], 'Not found in "Country" spreadsheet',
                   upper=True, also_blank=False)

    # Rules keyed off COUNTRY
    country_series = get("COUNTRY")
    if country_series is not None:
        country_upper = _clean(country_series).str.upper()
        non_us_mask = country_upper.ne("US")
        au_mask = country_upper.eq("AU")

        rid_type = get("RECIPIENTIDENTIFIERTYPE")
        if rid_type is not None:
            cleaned = _norm_series(rid_type)
            record("RECIPIENTIDENTIFIERTYPE", non_us_mask & ~cleaned.isin(lookups["identifier_type"]),
                   'COUNTRY is not "US" and value not found in "Identifier Type" spreadsheet')

        for col_name in ["RECIPIENTIDENTIFIERVALUE", "TRANSACTIONCONSENT", "COMPANYAGREEMENTID"]:
            mask = blank_mask(col_name)
            if mask is not None:
                record(col_name, non_us_mask & mask, 'COUNTRY is not "US" and value is blank')

        for col_name in ["ENGAGEMENTTYPE", "ENGAGEMENTNAME", "ENGAGEMENTDESCRIPTION"]:
            mask = blank_mask(col_name)
            if mask is not None:
                record(col_name, au_mask & mask, 'COUNTRY is "AU" and value is blank')

        for col_name in ["ENGAGEMENTSTARTDATE", "ENGAGEMENTENDDATE"]:
            series = get(col_name)
            if series is not None:
                cleaned = _clean(series)
                record(col_name, au_mask & cleaned.eq(""), 'COUNTRY is "AU" and value is blank')
                record(col_name, au_mask & cleaned.ne("") & ~cleaned.str.match(r"^[0-9]{8}$"),
                       'COUNTRY is "AU" and value is not in YYYYMMDD format')

    if secondary_purpose_nonblank is not None:
        record("PYMTCONTEXTINFO", secondary_purpose_nonblank, "Expected blank because SECONDARYPURPOSE has a value")

    # Roll the (column, message) -> count map up into one summary message per column
    per_column_messages = {}
    for (col_name, message), count in column_message_counts.items():
        entry = f"{message} ({count} row{'s' if count != 1 else ''})"
        per_column_messages.setdefault(col_name, []).append(entry)

    flag_row = {col: "" for col in df.columns}
    for col_name, messages in per_column_messages.items():
        flag_row[header_map[col_name]] = "; ".join(sorted(messages))

    if not any(flag_row.values()) and len(df.columns):
        flag_row[df.columns[0]] = "No checklist issues detected"

    return flag_row


In [11]:
def _norm_npi(value):
    """Normalize an NPI number for comparison (handles float-read values like 1234567890.0)."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    try:
        return str(int(float(text)))
    except ValueError:
        return text


def _load_roster_lookup(path):
    """Map (LASTNAME, FIRSTNAME) -> COMPANYSALESREPID from the Roster spreadsheet."""
    df = pd.read_excel(path)
    header_map = _header_map(df.columns)
    last = _clean(df[header_map["TRANSACTIONINITIATORLASTNAME"]]).str.upper()
    first = _clean(df[header_map["TRANSACTIONINITIATORFIRSTNAME"]]).str.upper()
    keys = last + "|" + first
    values = _clean(df[header_map["COMPANYSALESREPID"]])
    return dict(zip(keys, values))


def _load_profile_lookup(path):
    """Map USNPINUMBER -> COMPANYPROFILEID from the Profile Extract spreadsheet.

    When multiple rows share an NPI, prefer a row that actually has a
    COMPANYPROFILEID, then prefer an Active profile status.
    """
    df = pd.read_excel(path, sheet_name="Output")
    header_map = _header_map(df.columns)
    npi_keys = df[header_map["USNPINUMBER"]].apply(_norm_npi)
    profile_ids = _clean(df[header_map["COMPANYPROFILEID"]])
    status_col = header_map.get("PROFILE STATUS")
    is_active = _clean(df[status_col]).str.upper().eq("ACTIVE") if status_col else pd.Series(False, index=df.index)

    lookup_df = pd.DataFrame({
        "npi_key": npi_keys,
        "profile_id": profile_ids,
        "has_profile_id": profile_ids.ne(""),
        "is_active": is_active,
    })
    lookup_df = lookup_df[lookup_df["npi_key"] != ""]
    lookup_df = lookup_df.sort_values(by=["has_profile_id", "is_active"], ascending=[False, False])
    lookup_df = lookup_df.drop_duplicates(subset="npi_key", keep="first")
    return dict(zip(lookup_df["npi_key"], lookup_df["profile_id"]))


roster_lookup = _load_roster_lookup(lookup_paths["roster"])
profile_lookup = _load_profile_lookup(lookup_paths["profile_extract"])
print(f"Loaded {len(roster_lookup)} roster entries")
print(f"Loaded {len(profile_lookup)} profile extract NPI entries")


def _fill_company_sales_rep_id(df):
    """Fill blank COMPANYSALESREPID by matching TRANSACTIONINITIATOR first/last name against the Roster."""
    header_map = _header_map(df.columns)
    required = {"COMPANYSALESREPID", "TRANSACTIONINITIATORLASTNAME", "TRANSACTIONINITIATORFIRSTNAME"}
    if not required.issubset(header_map):
        return
    rep_col = header_map["COMPANYSALESREPID"]
    last = _clean(df[header_map["TRANSACTIONINITIATORLASTNAME"]]).str.upper()
    first = _clean(df[header_map["TRANSACTIONINITIATORFIRSTNAME"]]).str.upper()
    matched = (last + "|" + first).map(roster_lookup)
    fill_mask = _clean(df[rep_col]).eq("") & matched.notna()
    if fill_mask.any():
        df[rep_col] = df[rep_col].astype(object)
        df.loc[fill_mask, rep_col] = matched[fill_mask]


def _fill_company_profile_id(df):
    """Fill blank COMPANYPROFILEID by matching USNPINUMBER against the Profile Extract."""
    header_map = _header_map(df.columns)
    required = {"COMPANYPROFILEID", "USNPINUMBER"}
    if not required.issubset(header_map):
        return
    profile_col = header_map["COMPANYPROFILEID"]
    matched = df[header_map["USNPINUMBER"]].apply(_norm_npi).map(profile_lookup)
    fill_mask = _clean(df[profile_col]).eq("") & matched.notna()
    if fill_mask.any():
        df[profile_col] = df[profile_col].astype(object)
        df.loc[fill_mask, profile_col] = matched[fill_mask]


VIRTUAL_VENUE_MAP = {
    "VENUECITY": "CITY",
    "VENUEPROVINCE": "PROVINCE",
    "VENUEPOSTALCODE": "POSTALCODE",
    "VENUECOUNTRY": "COUNTRY",
}


def _apply_transformations(df, file_name):
    """Clean-up steps run before validation. Only touches columns that exist in the file."""
    header_map = _header_map(df.columns)

    def col(name):
        return header_map.get(name)

    def set_values(name, mask, values):
        c = col(name)
        if c is None or not mask.any():
            return
        df[c] = df[c].astype(object)
        df.loc[mask, c] = values[mask] if isinstance(values, pd.Series) else values

    # PRODUCT: uppercase so it matches the Product spreadsheet regardless of case
    c = col("PRODUCT")
    if c:
        set_values("PRODUCT", _clean(df[c]).ne(""), _norm_series(df[c]).str.upper())

    # COUNTRY / RECIPIENTIDENTIFIERCOUNTRY: "USA" -> "US"
    for name in ["COUNTRY", "RECIPIENTIDENTIFIERCOUNTRY"]:
        c = col(name)
        if c:
            set_values(name, _clean(df[c]).str.upper().eq("USA"), "US")

    # VENUE*: replace "Virtual" with the recipient's address fields
    # (runs after the USA -> US fix so VENUECOUNTRY picks up "US")
    for venue_col, source_col in VIRTUAL_VENUE_MAP.items():
        vc, sc = col(venue_col), col(source_col)
        if vc is None or sc is None:
            continue
        set_values(venue_col, _clean(df[vc]).str.upper().eq("VIRTUAL"), df[sc])

    # PYMTCONTEXTINFO: if blank, COMPANYEVENTID-TRANSACTIONDATE-ENGAGEMENTNAME
    pc = col("PYMTCONTEXTINFO")
    if pc:
        parts = [
            _clean(df[col(n)]) if col(n) else pd.Series("", index=df.index)
            for n in ["COMPANYEVENTID", "TRANSACTIONDATE", "ENGAGEMENTNAME"]
        ]
        combined = pd.Series(
            ["-".join(v for v in vals if v) for vals in zip(*parts)],
            index=df.index, dtype=object,
        )
        mask = _clean(df[pc]).eq("") & combined.ne("")
        sp = col("SECONDARYPURPOSE")
        if sp:
            # Checklist requires PYMTCONTEXTINFO blank when SECONDARYPURPOSE has a value
            mask &= _clean(df[sp]).eq("")
        set_values("PYMTCONTEXTINFO", mask, combined)

    # COMMENTS: if blank, use the original source file name
    cc = col("COMMENTS")
    if cc:
        set_values("COMMENTS", _clean(df[cc]).eq(""), file_name)


Loaded 202 roster entries
Loaded 12071 profile extract NPI entries


In [13]:


files = sorted(
    list(upload_dir.glob("*.xlsx")) +
    list(upload_dir.glob("*.csv"))
)
processed = []
skipped = []

for file_path in files:
    try:
        if file_path.suffix.lower() == ".csv":
            sheets = {
                file_path.stem: pd.read_csv(file_path, dtype=str)
            }
        else:
            sheets = pd.read_excel(
                file_path,
                sheet_name=None,
                engine="openpyxl",
                dtype=str
            )

    except Exception as exc:
        skipped.append((file_path.name, str(exc)))
        continue

    for sheet_name, df in sheets.items():
        if len(sheets) == 1:
            out_name = f"{output_prefix}_{file_path.stem}.csv"
        else:
            safe_sheet = re.sub(r"[^A-Za-z0-9_-]+", "_", sheet_name).strip("_") or "Sheet1"
            out_name = f"{output_prefix}_{file_path.stem}_{safe_sheet}.csv"

        out_path = working_dir / out_name

        if len(df.columns) == 0:
            with open(out_path, "w", encoding="utf-8-sig", newline="") as f:
                f.write("Issue\nSheet is empty\n")
            processed.append((file_path.name, out_name, 0))
            continue

        df = df.copy()
        _fill_company_sales_rep_id(df)
        _fill_company_profile_id(df)
        _apply_transformations(df, file_path.name)

        flag_row = _flag_columns_for_df(df, lookups)
        flagged_columns = sum(1 for v in flag_row.values() if v and v != "No checklist issues detected")

        with open(out_path, "w", encoding="utf-8-sig", newline="") as f:
            pd.DataFrame([flag_row], columns=df.columns).to_csv(f, index=False, header=False)
            df.to_csv(f, index=False, header=True)

        processed.append((file_path.name, out_name, flagged_columns))

print(f"Processed {len(processed)} files")
for name, out_name, flagged_columns in processed:
    print(f"- {name} -> {out_name} ({flagged_columns} column(s) flagged)")

if skipped:
    print("\nSkipped files:")
    for name, error in skipped:
        print(f"- {name}: {error}")

print(f"\nWorking files saved to: {working_dir}")


Processed 2 files
- Impact_07.15.26_ACT-2026-ADB-020_Exelixis Zanza CRC NAB Virtual.csv -> SPHASE_Impact_07.15.26_ACT-2026-ADB-020_Exelixis Zanza CRC NAB Virtual.csv (16 column(s) flagged)
- Impact_07.27.26_ACT-2026-ADB-030_Exelixis XB010 Ophtho ECP Offline Review.csv -> SPHASE_Impact_07.27.26_ACT-2026-ADB-030_Exelixis XB010 Ophtho ECP Offline Review.csv (14 column(s) flagged)

Working files saved to: C:\Users\cmclaren\OneDrive - Exelixis Inc\Desktop\Transparency\US\Q3 2026\BiomedCom\Working Files


In [18]:
if processed:
    overview = pd.DataFrame(processed, columns=["Source File", "Output File", "Flagged Columns"])
    overview
else:
    print("No files were processed.")
